# Generate DNA sequence from Amino Acid Sequence for TwistBio

## Setup

### Install Packages

In [2]:
%pip install -q dnachisel biopython requests pandas

Note: you may need to restart the kernel to use updated packages.


### Imports

In [3]:
import os
import itertools
import pandas as pd
from Bio import Restriction
from dnachisel import (
    DnaOptimizationProblem,
    reverse_translate,
    CodonOptimize,
    AvoidPattern,
    EnforceTranslation,
    EnzymeSitePattern
)

from Bio import SeqIO
from Bio import Align
from Bio.Seq import Seq
from Bio.Align import MultipleSeqAlignment
from Bio.SeqRecord import SeqRecord
from io import StringIO

### Options

In [4]:
# "combinatorial" = Multiply (Loop1_VarA + Loop2_VarA, Loop1_VarA + Loop2_VarB...)
# "positional"    = Add      (Loop1_VarA + Loop2_WT, Loop1_WT + Loop2_VarA...)
GENERATION_MODE = "positional"

# Your Input Data
CANONICAL_SEQ = "CSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEASESLCGLKLEVNKYQYLLTGRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCNCKIKSCYYLPCFVTSKNECLWTDMLSNFGYPGYQSKHYACIRQKGGYCSWYRGWAPPDKSIINATDP" # TIMP3
GENE_NAME = "TIMP3YeastGene_01"

#Twist API Credentials (You must request these from Twist)
API_BASE_URL = "https://twist-api.twistbioscience.com/v1"
API_TOKEN = "YOUR_TWIST_API_TOKEN"
USER_EMAIL = "your_email@example.com"

# 3. Optimization Settings
HOST_ORGANISM = "s_cerevisiae" 
PROJECT_NAME = "Yeast_Loop_Library_2025"

OUTPUT_FOLDER = "../Local/Twist_Order_Dec2025"

# Loops are positioned 2 AA down from normal because of the removal of "CT" from the canonical seq
DESIGN_SPECS = {
    'AB_LOOP': {
        'range': (28, 34), 
        'variants': ["tlpdgske", "knpdgtlt", "kgpyge", "patptstrgaggee", "eversghkvke", "tdtfptanwtgev", "dgptge"] 
    },
    'C_LOOP': {
        'range': (60, 66), 
        'variants': ["asgpitvngetiw", "ltqeelpdpnavspc", "sveslc", "asveavetgfs", "anpeyc", "ggnygsck"]
    }
}

# Format: {"name": "Your_Name", "seq": "FULL_AA_SEQUENCE"}
CUSTOM_SEQUENCES = [
    {
        "name": "Var_ABC_LOOP-KNPDGTLT_ANPEYC", 
        "seq": "CSPSHPQDAFCNSDIVIRAKVVGKKLVKKNPDGTLTLVYTIKQMKMYRGFTKMPHVQYIHTEANPEYCLKLEVNKYQYLLTGRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCNCKIKSCYYLPCFVTSKNECLWTDMLSNFGYPGYQSKHYACIRQKGGYCSWYRGWAPPDKSIINATDP"
    },
    {
        "name": "Var_ABC_LOOP-EVERSGHKVKE_LTQEELPDPNAVSPC", 
        "seq": "CSPSHPQDAFCNSDIVIRAKVVGKKLVKEVERSGHKVKELVYTIKQMKMYRGFTKMPHVQYIHTELTQEELPDPNAVSPCLKLEVNKYQYLLTGRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCNCKIKSCYYLPCFVTSKNECLWTDMLSNFGYPGYQSKHYACIRQKGGYCSWYRGWAPPDKSIINATDP"
    },
    {
        "name": "Var_ABC_LOOP-KGPYGE_SVESLC", 
        "seq": "CSPSHPQDAFCNSDIVIRAKVVGKKLVKKGPYGELVYTIKQMKMYRGFTKMPHVQYIHTESVESLCLKLEVNKYQYLLTGRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCNCKIKSCYYLPCFVTSKNECLWTDMLSNFGYPGYQSKHYACIRQKGGYCSWYRGWAPPDKSIINATDP"
    },
]

# List of "Known Endonucleases" to avoid (Common cloning sites + Golden Gate)
RESTRICTION_SITES_TO_AVOID = [
    "BsrGI_site", "BamHI_site",
    "BsaI_site" # Added good practice for Golden Gate compatibility
]

### Functions

In [5]:
def setup_output_folder(folder_path):
    """Creates the output directory if it doesn't exist."""
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Created output directory: {folder_path}")
    else:
        print(f"Using existing output directory: {folder_path}")

def get_enzyme_name(site_string):
    """Strips '_site' suffix to get standard enzyme name."""
    return site_string.replace("_site", "")

def get_enzyme_constraint(site_name):
    """
    Converts user-friendly names (e.g., 'BsrGI_site') 
    into DNA Chisel constraints.
    """
    clean_name = site_name.replace("_site", "")
    # EnzymeSitePattern looks up the cut sequence for the enzyme name
    return AvoidPattern(EnzymeSitePattern(clean_name))

def get_enzyme_sequence(site_string):
    """
    Looks up the actual cut sequence using Biopython.
    Used for the Validation Report.
    """
    name = get_enzyme_name(site_string)
    try:
        # Dynamic lookup from Biopython's Restriction library
        enzyme_obj = getattr(Restriction, name)
        return enzyme_obj.site
    except AttributeError:
        print(f"CRITICAL WARNING: Enzyme '{name}' not found in Biopython database.")
        return "NNNNNN" # Fail-safe

In [6]:
def generate_combinations(canonical, specs, mode="combinatorial"):
    """
    Generates all combinatorial variants.
    """
    loop_names = sorted(specs.keys())
    combined_designs = []

    # Ensure canonical is upper case
    canonical = canonical.upper()
    
    # ---------------------------------------
    # MODE A: COMBINATORIAL (Cartesian Product)
    # ---------------------------------------
    if mode == "combinatorial":
        print(f"   > Mode: Combinatorial (Multiplying variants)")
        lists_of_variants = [specs[name]['variants'] for name in loop_names]
        
        for combination in itertools.product(*lists_of_variants):
            current_seq = list(canonical)
            variant_tags = []
            
            # Sort reverse to preserve indices
            sorted_loops = sorted(loop_names, key=lambda x: specs[x]['range'][0], reverse=True)
            
            for loop_name in sorted_loops:
                start, end = specs[loop_name]['range']
                # Find the variant for this loop
                val = next(v for v in combination if v in specs[loop_name]['variants'])
                val = val.upper()
                
                current_seq[start:end] = list(val)
                variant_tags.append(f"{loop_name}-{val}")
                
            final_aa_seq = "".join(current_seq)
            # Name: Var_LOOP1-VAL_LOOP2-VAL
            variant_name = f"Var_{'_'.join(variant_tags[::-1])}"
            
            combined_designs.append({ "name": variant_name, "aa_sequence": final_aa_seq })

    # ---------------------------------------
    # MODE B: POSITIONAL (One Loop at a Time)
    # ---------------------------------------
    elif mode == "positional":
        print(f"   > Mode: Positional (Adding variants, others stay WT)")
        
        for loop_name in loop_names:
            variants = specs[loop_name]['variants']
            start, end = specs[loop_name]['range']
            
            for val in variants:
                val = val.upper()
                current_seq = list(canonical)
                
                # Apply ONLY the current loop variant
                # All other loops remain as they are in CANONICAL_SEQ (Wild Type)
                current_seq[start:end] = list(val)
                
                final_aa_seq = "".join(current_seq)
                
                # Name: Var_LOOP1-VAL (Implies others are WT)
                variant_name = f"Var_{loop_name}-{val}"
                
                combined_designs.append({ "name": variant_name, "aa_sequence": final_aa_seq })
                
    else:
        raise ValueError("GENERATION_MODE must be 'combinatorial' or 'positional'")

    return combined_designs

def optimize_variant(design_entry):
    print(f"  > Optimizing codons for {HOST_ORGANISM}...") # Optional noise
    print(f"Optimizing {design_entry['name']}...")
    
    naive_dna = reverse_translate(design_entry['aa_sequence'])
    
    constraints = [EnforceTranslation()]
    
    # Add named restriction sites dynamically
    for site in RESTRICTION_SITES_TO_AVOID:
        try:
            constraints.append(get_enzyme_constraint(site))
        except Exception as e:
            print(f"Warning: Could not process restriction site {site}. {e}")

    # Standard synthesis constraints
    constraints.extend([
        AvoidPattern("9xA"), AvoidPattern("9xT"),
        AvoidPattern("9xG"), AvoidPattern("9xC")
    ])
    
    problem = DnaOptimizationProblem(
        sequence=naive_dna,
        constraints=constraints,
        objectives=[CodonOptimize(species=HOST_ORGANISM)],
        logger=None
    )
    
    problem.resolve_constraints()
    problem.optimize()
    
    return problem.sequence

def validate_and_report(variants_data, folder):
    """
    Look up sequences for the sites in RESTRICTION_SITES_TO_AVOID
    and verify they are absent from the optimized DNA.
    """
    report_path = os.path.join(folder, "validation_report.txt")
    print(f"\n--- Running Validation Checks ---")
    
    with open(report_path, "w") as f:
        f.write("VALIDATION REPORT\n")
        f.write("=================\n")
        
        # 1. Resolve patterns once
        patterns = {}
        for site in RESTRICTION_SITES_TO_AVOID:
            seq = get_enzyme_sequence(site)
            patterns[get_enzyme_name(site)] = seq
            f.write(f"Avoid Target: {get_enzyme_name(site)} = {seq}\n")
        
        f.write("\n")
        
        passes = 0
        fails = 0
        
        for v in variants_data:
            dna = v['dna']
            issues = []
            
            # Check for enzyme sites
            for enzyme, pattern in patterns.items():
                if pattern in dna:
                    issues.append(f"FAIL: Found {enzyme} ({pattern})")
            
            # Check for homopolymers
            for base in ['A', 'T', 'G', 'C']:
                if base * 10 in dna:
                    issues.append(f"WARN: Homopolymer run {base}x10")

            status = "PASS" if not issues else "FAIL"
            if status == "PASS": passes += 1
            else: fails += 1
            
            f.write(f"[{status}] {v['name']}\n")
            if issues:
                for issue in issues:
                    f.write(f"       !!! {issue}\n")
            f.write("-" * 50 + "\n")
            
        summary = f"SUMMARY: {passes} Passed, {fails} Failed."
        f.write(f"\n{summary}")
        print(summary)

def save_output_files(data_list, folder):
    # Paths
    fasta_path = os.path.join(folder, "twist_library.fasta")
    csv_path = os.path.join(folder, "twist_library.csv")

    # Save FASTA
    with open(fasta_path, "w") as f:
        for seq in data_list:
            f.write(f">{seq['name']}\n")
            f.write(f"{seq['dna']}\n")
    
    # Save CSV
    df = pd.DataFrame(data_list)
    df = df[["name", "dna", "aa_seq"]] 
    df.columns = ["Construct Name", "DNA Sequence", "Amino Acid Sequence"]
    df.to_csv(csv_path, index=False)
    
    print(f"Saved FASTA: {fasta_path}")
    print(f"Saved CSV:   {csv_path}")

def upload_draft_to_twist(sequences):
    """
    Uploads sequences as a 'Draft' project (Design phase).
    Does NOT execute an order.
    """
    print(f"\n--- Uploading {len(sequences)} sequences to Twist (Draft) ---")
    
    url = f"{API_BASE_URL}/constructs" 
    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    }
    
    # Twist API accepts batch creation
    # We set properties to ensure it remains a draft/design object
    payload_items = []
    for s in sequences:
        payload_items.append({
            "name": s['name'],
            "sequences": [s['dna']],
            "type": "CLONED_GENE", # Change to NON_CLONED_GENE if just fragments
            # "vector_mes_uid": "GET_THIS_FROM_PORTAL", 
            # "insertion_point_mes_uid": "GET_THIS_FROM_PORTAL"
        })

    # Note: Real API interaction requires handling batch limits (usually 500 items/call)
    # and valid Vector UIDs. This is a structural example.
    print(f"Prepared payload for {len(payload_items)} items.")
    print("Status: Ready to Send (Commented out for safety)")
    
    # response = requests.post(url, headers=headers, json=payload_items)
    # if response.status_code == 200:
    #    print("Upload Success! Sequences are now in your Twist account.")
    # else:
    #    print(f"Upload Failed: {response.text}")
    """
    Submits the optimized sequence to Twist Bioscience API.
    Note: This requires a valid 'Vector ID' and 'Insertion Point ID' 
    which you must fetch from your Twist account if cloning into a vector.
    """
    print(f"\n--- Preparing Submission for {gene_name} ---")
    
    # Payload structure based on Twist API documentation for Clonal Genes
    # NOTE: You normally need to fetch your specific 'vector_uid' first.
    payload = {
        "name": gene_name,
        "type": "CLONED_GENE", # or "NON_CLONED_GENE" for fragments
        "sequences": [dna_seq],
        "adapters_on": False,
        # "vector_mes_uid": "YOUR_VECTOR_ID_HERE",  <-- REQUIRED for Clonal Genes
        # "insertion_point_mes_uid": "YOUR_INSERTION_ID_HERE" <-- REQUIRED for Clonal Genes
    }

    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "X-End-User-Token": API_TOKEN,
        "Content-Type": "application/json"
    }

    # API Endpoint for creating a construct
    url = f"{API_BASE_URL}/users/{USER_EMAIL}/constructs/"

    try:
        # Uncomment the line below to actually send the request
        # response = requests.post(url, headers=headers, json=payload)
        # response.raise_for_status()
        
        # Simulating a successful response for this demo
        print("Payload ready for submission:")
        print(json.dumps(payload, indent=2))
        print("\n(Actual submission commented out. Add your API Token to enable.)")
        
    except requests.exceptions.RequestException as e:
        print(f"Error submitting to Twist: {e}")

## Generate DNA and AA Sequences

In [7]:
setup_output_folder(OUTPUT_FOLDER)

# Generate
print("--- 1. Generating Variants ---")
variants = generate_combinations(CANONICAL_SEQ, DESIGN_SPECS, mode=GENERATION_MODE)

print(f"   > Adding {len(CUSTOM_SEQUENCES)} custom sequences...")
for cust in CUSTOM_SEQUENCES:
    variants.append({
        "name": cust["name"],
        "aa_sequence": cust["seq"].upper() # Ensure upper case
    })

final_output = []

# Optimize
for var in variants:
    opt_dna = optimize_variant(var)
    final_output.append({
        "name": var['name'],
        "aa_seq": var['aa_sequence'],
        "dna": opt_dna
    })
    print(f"{var['name']:<40} | Optimized")
print(f"{'VARIANT NAME':<40} | {'STATUS'}")
print("-" * 60)

# Validate & Save (Targeting the Output Folder)
validate_and_report(final_output, OUTPUT_FOLDER)
save_output_files(final_output, OUTPUT_FOLDER)

Using existing output directory: ../Local/Twist_Order_Dec2025
--- 1. Generating Variants ---
   > Mode: Positional (Adding variants, others stay WT)
   > Adding 3 custom sequences...
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-TLPDGSKE...
Var_AB_LOOP-TLPDGSKE                     | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-KNPDGTLT...
Var_AB_LOOP-KNPDGTLT                     | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-KGPYGE...
Var_AB_LOOP-KGPYGE                       | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-PATPTSTRGAGGEE...
Var_AB_LOOP-PATPTSTRGAGGEE               | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-EVERSGHKVKE...
Var_AB_LOOP-EVERSGHKVKE                  | Optimized
  > Optimizing codons for s_cerevisiae...
Optimizing Var_AB_LOOP-TDTFPTANWTGEV...
Var_AB_LOOP-TDTFPTANWTGEV                | Optimized
  > Optimizing codons for

## Gene + Plasmid Gene Databank Test

### Options

In [38]:
# The fixed sequences from the pCHA plasmid
UPSTREAM_SEQ = "MKVLIVLLAIFAALPLALAQPVISTTVGSAAEGSLDKRCT"
DOWNSTREAM_SEQ = "GSTRAEQKLISEEDLQELTTICEQIPSPTLESTPYSLSTTTILANGKAMQGVFEYYKSVTFVSNCGSHPSTTSKGSPINTQYVF*"

FULL_CANONICAL = UPSTREAM_SEQ + CANONICAL_SEQ.rstrip('*') + DOWNSTREAM_SEQ

print(f"Full Canonical Length: {len(FULL_CANONICAL)} aa")
print(f"Upstream Region:       1 - {len(UPSTREAM_SEQ)}")
print(f"Insert Region:         {len(UPSTREAM_SEQ)+1} - {len(UPSTREAM_SEQ)+len(CANONICAL_SEQ)}")
print(f"Downstream Region:     {len(UPSTREAM_SEQ)+len(CANONICAL_SEQ)+1} - {len(FULL_CANONICAL)}")

# Format: {'Variant_Name': 'EXPECTED_AA_SEQUENCE'}
expected_variants = {
    v['name']: v['aa_sequence'].rstrip('*') 
    for v in variants
}

# Path to your folder containing Twist .gb files
gb_folder_path = OUTPUT_FOLDER + "/Draft"

Full Canonical Length: 311 aa
Upstream Region:       1 - 40
Insert Region:         41 - 226
Downstream Region:     227 - 311


### Options

In [39]:
def find_best_match_circular(file_path, expected_full_aa):
    """
    Handles circular plasmids by doubling the sequence. 
    Performs 6-frame translation and local alignment.
    """
    # 1. Read File
    with open(file_path, 'r') as f:
        lines = f.readlines()
    if lines and lines[0].startswith("LOCUS"):
        parts = lines[0].split()
        if 'bp' in parts:
            length = parts[parts.index('bp')-1]
            lines[0] = f"LOCUS       Twist_Import            {length} bp    DNA     circular SYN 01-JAN-2025\n"
    
    handle = StringIO("".join(lines))
    record = SeqIO.read(handle, "genbank")
    
    # --- FIX: DOUBLE THE SEQUENCE ---
    # This solves the wrap-around truncation issue
    dna = record.seq + record.seq
    
    # 2. Generate 6 Frames
    frames = {}
    frames['Fwd_1'] = str(dna.translate(to_stop=False))
    frames['Fwd_2'] = str(dna[1:].translate(to_stop=False))
    frames['Fwd_3'] = str(dna[2:].translate(to_stop=False))
    
    rc_dna = dna.reverse_complement()
    frames['Rev_1'] = str(rc_dna.translate(to_stop=False))
    frames['Rev_2'] = str(rc_dna[1:].translate(to_stop=False))
    frames['Rev_3'] = str(rc_dna[2:].translate(to_stop=False))

    # 3. Local Alignment Search
    best_score = -1
    best_seq_found = None
    
    aligner = Align.PairwiseAligner()
    aligner.mode = 'local' 
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5
    
    clean_expected = expected_full_aa.replace('*', '')

    for frame_name, frame_seq in frames.items():
        score = aligner.score(frame_seq, clean_expected)
        
        if score > best_score:
            best_score = score
            
            # Extract the aligned region
            alignment = aligner.align(frame_seq, clean_expected)[0]
            target_segments = alignment.aligned[0]
            
            if len(target_segments) > 0:
                start_idx = target_segments[0][0]
                end_idx = target_segments[-1][1]
                best_seq_found = frame_seq[start_idx:end_idx]

    if best_seq_found is None:
        return None, "FAIL: No Alignment Found"
        
    # Check length (allow small gaps/indels but not massive truncation)
    if len(best_seq_found) < len(clean_expected) * 0.95:
        return best_seq_found, "FAIL: Sequence Truncated (Tag Missing?)"

    return best_seq_found, "SUCCESS"

def print_smart_stacked_alignment(reference_seq, variants_df):
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5

    # Filter only passing variants
    valid_df = variants_df[variants_df['QC_Pass'] == True]
    
    if valid_df.empty:
        print("No verified variants to align.")
        return

    alignment_groups = {}
    print(f"Aligning {len(valid_df)} passing variants...")

    # --- GROUPING PHASE ---
    for index, row in valid_df.iterrows():
        var_seq = row['Found_AA'].rstrip('*')
        name = row['Variant']
        
        # Get the first (best) alignment
        alignment = aligner.align(reference_seq, var_seq)[0]
        
        # --- THE FIX: DIRECT ACCESS ---
        # In modern Biopython, alignment[0] is the gapped Target
        # and alignment[1] is the gapped Query. No parsing needed!
        ref_gapped = str(alignment[0]) # Canonical with dashes
        var_gapped = str(alignment[1]) # Variant with dashes
        
        # Group by the Reference Pattern 
        if ref_gapped not in alignment_groups:
            alignment_groups[ref_gapped] = []
        
        alignment_groups[ref_gapped].append({
            'name': name,
            'seq': var_gapped
        })

    # --- PRINTING PHASE ---
    group_counter = 1
    total_groups = len(alignment_groups)
    
    # Sort groups by length (shortest first usually looks cleaner)
    sorted_groups = sorted(alignment_groups.items(), key=lambda x: len(x[0]))
    
    for ref_pattern, vars_in_group in sorted_groups:
        print("\n" + "="*80)
        print(f"ALIGNMENT GROUP {group_counter}/{total_groups}")
        print(f"Variants in this group: {len(vars_in_group)}")
        print("="*80)
        
        chunk_size = 150
        
        for i in range(0, len(ref_pattern), chunk_size):
            chunk_end = min(i + chunk_size, len(ref_pattern))
            
            print(f"\nRegion {i+1}-{chunk_end}:")
            print(f"{'FULL_CANONICAL':<25} {ref_pattern[i:chunk_end]}")
            print("-" * (26 + (chunk_end - i)))
            
            for v in vars_in_group:
                name = v['name']
                seq = v['seq']
                display_name = (name[:22] + '..') if len(name) > 22 else name
                
                visual_string = ""
                for j in range(i, chunk_end):
                    if j < len(seq):
                        ref_char = ref_pattern[j]
                        var_char = seq[j]
                        
                        if var_char == ref_char:
                            visual_string += "."
                        elif var_char == '-':
                            visual_string += "-" 
                        elif ref_char == '-':
                            # Variant has insertion relative to canonical
                            visual_string += var_char 
                        else:
                            # Mismatch
                            visual_string += var_char 
                    else:
                        visual_string += " "
                
                print(f"{display_name:<25} {visual_string}")
        
        group_counter += 1

def print_unified_alignment(reference_seq, variants_df):
    aligner = Align.PairwiseAligner()
    aligner.mode = 'global'
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5

    valid_df = variants_df[variants_df['QC_Pass'] == True]
    if valid_df.empty:
        print("No verified variants to align.")
        return

    print(f"Constructing Unified Alignment for {len(valid_df)} variants...")

    # --- DATA STRUCTURES ---
    # We view the canonical sequence as a series of "Slots" (indices 0 to N).
    # Each slot has:
    # 1. The Canonical Residue
    # 2. A "Bucket" for insertions occurring immediately AFTER this residue.
    # 3. A list of what every variant does at this residue.
    
    # Initialize Slots
    # size is len + 1 to handle insertions at the very start (N-term)
    slots = [{'ref_aa': '', 'max_insert': 0} for _ in range(len(reference_seq) + 1)]
    
    # Fill in the Ref AA (Slot 0 is "Start", Slot 1 is Ref[0], etc.)
    for i, char in enumerate(reference_seq):
        slots[i+1]['ref_aa'] = char

    # Store parsed variant data here
    variant_data_list = []

    # --- PASS 1: MAP EVERY VARIANT TO THE SLOTS ---
    for index, row in valid_df.iterrows():
        var_seq = row['Found_AA'].rstrip('*')
        name = row['Variant']
        
        # Get Alignment
        alignment = aligner.align(reference_seq, var_seq)[0]
        ref_gapped = str(alignment[0])
        var_gapped = str(alignment[1])
        
        # Parse the alignment to map to our slots
        # We walk through the GAPPED reference.
        # current_slot corresponds to the index in the UNGAPPED reference.
        current_slot = 0 
        
        # Temp storage for this specific variant
        # Format: { slot_index: {'char': 'X', 'insertion': 'STR'} }
        var_map = {}
        
        # We accumulate insertion strings character by character
        active_insertion = ""
        
        # Iterate through the aligned strings
        for r_char, v_char in zip(ref_gapped, var_gapped):
            
            if r_char != '-':
                # We have hit a canonical backbone residue.
                # 1. Close out any active insertion from the PREVIOUS slot
                if active_insertion:
                    prev_len = len(active_insertion)
                    if prev_len > slots[current_slot]['max_insert']:
                        slots[current_slot]['max_insert'] = prev_len
                    
                    # Store the insertion for the variant
                    if current_slot not in var_map: var_map[current_slot] = {'char': '', 'insertion': ''}
                    var_map[current_slot]['insertion'] = active_insertion
                    active_insertion = ""

                # 2. Move to next slot
                current_slot += 1
                
                # 3. Record the variant's behavior at this backbone position
                if current_slot not in var_map: var_map[current_slot] = {'char': '', 'insertion': ''}
                
                if v_char == '-':
                    var_map[current_slot]['char'] = '-' # Deletion
                else:
                    var_map[current_slot]['char'] = v_char # Match or Mismatch

            else:
                # r_char is '-', meaning valid sequence is inserted here relative to canonical
                active_insertion += v_char

        # Catch trailing insertions (C-terminus)
        if active_insertion:
            if len(active_insertion) > slots[current_slot]['max_insert']:
                slots[current_slot]['max_insert'] = len(active_insertion)
            if current_slot not in var_map: var_map[current_slot] = {'char': '', 'insertion': ''}
            var_map[current_slot]['insertion'] = active_insertion

        variant_data_list.append({'name': name, 'map': var_map})

    # --- PASS 2: CONSTRUCT FINAL STRINGS ---
    # Now we flatten the slots into strings, adding padding where 'max_insert' demands it.
    
    final_canonical = ""
    final_variants = {v['name']: "" for v in variant_data_list}
    
    # Iterate through all slots (0 to N)
    for i, slot in enumerate(slots):
        # 1. The Backbone Character (skip for slot 0 as it's purely N-term insertion holder)
        if i > 0:
            final_canonical += slot['ref_aa']
            for v in variant_data_list:
                # Add the char tracked for this slot, or assume match/error? 
                # Actually we tracked it explicitly.
                v_entry = v['map'].get(i, {'char': '?', 'insertion': ''})
                
                # Visual logic:
                if v_entry['char'] == slot['ref_aa']:
                    final_variants[v['name']] += "." # Match
                elif v_entry['char'] == '-':
                    final_variants[v['name']] += "-" # Deletion
                elif v_entry['char'] == '?':
                    final_variants[v['name']] += "?" # Should not happen
                else:
                    final_variants[v['name']] += v_entry['char'] # Mismatch

        # 2. The Insertion Space (if any variant has an insertion here)
        gap_width = slot['max_insert']
        if gap_width > 0:
            final_canonical += "-" * gap_width
            for v in variant_data_list:
                v_entry = v['map'].get(i, {'char': '', 'insertion': ''})
                ins_seq = v_entry['insertion']
                # Pad the insertion to the max width
                final_variants[v['name']] += ins_seq.ljust(gap_width, '-')

    # --- PASS 3: PRINTING ---
    print("\n" + "="*80)
    print(f"UNIFIED ALIGNMENT (All Variants vs Canonical)")
    print("="*80)
    
    chunk_size = 150
    ref_str = final_canonical
    
    for i in range(0, len(ref_str), chunk_size):
        chunk_end = min(i + chunk_size, len(ref_str))
        
        print(f"\nRegion {i+1}-{chunk_end}:")
        print(f"{'FULL_CANONICAL':<25} {ref_str[i:chunk_end]}")
        print("-" * (26 + (chunk_end - i)))
        
        # Sort variants by name for consistent order
        for v in sorted(variant_data_list, key=lambda x: x['name']):
            name = v['name']
            seq_segment = final_variants[name][i:chunk_end]
            
            display_name = (name[:22] + '..') if len(name) > 22 else name
            print(f"{display_name:<25} {seq_segment}")        

## Run Analysis

### Check DNA Sequences from `*.gb` files

In [46]:
results = []
verified_records = [] # Store for the final diagram

for filename in os.listdir(gb_folder_path):
    if filename.endswith(".gb") or filename.endswith(".gbk"):
        file_path = os.path.join(gb_folder_path, filename)
        
        matched_name = next((name for name in expected_variants if name in filename), None)
        
        if matched_name:
            variant_core = expected_variants[matched_name]
            full_expected = UPSTREAM_SEQ + variant_core + DOWNSTREAM_SEQ
            
            # Find Sequence (Circular Safe)
            found_aa, status_extraction = find_best_match_circular(file_path, full_expected)
            
            if found_aa is None:
                 results.append({'Variant': matched_name, 'Filename': filename, 'Status': status_extraction, 'QC_Pass': False})
                 continue

            # Checks
            # Check for internal stops (ignoring end)
            stop_status = "PASS"
            if '*' in found_aa[:-1]: stop_status = "FAIL: Internal Stop Codon"
            
            # Exact Match Check
            clean_found = found_aa.replace('*', '')
            clean_expected = full_expected.replace('*', '')
            exact_match = (clean_found == clean_expected)
            
            # Final Status
            qc_pass = True
            final_status = "OK"
            
            if "FAIL" in status_extraction: # Catch truncation warnings
                qc_pass = False; final_status = status_extraction
            elif "FAIL" in stop_status:
                qc_pass = False; final_status = stop_status
            elif not exact_match:
                qc_pass = False; final_status = "FAIL: Sequence Mismatch"
            
            # Alignment Object for failures
            aligner = Align.PairwiseAligner()
            aligner.mode = 'global'
            alignment_vis = aligner.align(clean_found, clean_expected)[0] if not exact_match else None

            # Store for results
            results.append({
                'Variant': matched_name, 'Filename': filename, 
                'Status': final_status, 'QC_Pass': qc_pass, 
                'Found_AA': str(found_aa), 'Alignment_Object': alignment_vis
            })
            
            # Store for Multi-Alignment Diagram (Only if passing or near-passing)
            if qc_pass:
                # Create a SeqRecord for the MSA
                # We label it with the variant name
                rec = SeqRecord(Seq(clean_found), id=matched_name, description="")
                verified_records.append(rec)

        else:
            results.append({'Variant': matched_name, 'Filename': filename, 'Status': "No match Found", 'QC_Pass': False})

df = pd.DataFrame(results)

C:\Users\ryangustafson\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\Bio\Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


### Show Prelimiary Sequence Analysis Results

In [41]:
if not df.empty:
    print(f"Total: {len(df)} | Pass: {len(df[df['QC_Pass']==True])} | Fail: {len(df[df['QC_Pass']==False])}")
    
    failures = df[df['QC_Pass'] == False]
    if not failures.empty:
        print("\n=== 🚩 FAILURE REPORT ===")
        for i, row in failures.iterrows():
            print(f"\nVariant: {row['Variant']}\nStatus:  {row['Status']}")
            if row['Alignment_Object']: print(row['Alignment_Object'])
    
    def color_pass_fail(val):
        return f'color: {"green" if val else "red"}; font-weight: bold'
    display(df[['Variant', 'Filename', 'Status', 'QC_Pass']].style.map(color_pass_fail, subset=['QC_Pass']))

Total: 17 | Pass: 15 | Fail: 2

=== 🚩 FAILURE REPORT ===

Variant: None
Status:  No match Found
nan

Variant: None
Status:  No match Found
nan


,Variant,Filename,Status,QC_Pass
0,None,pET-SC-EmGFP-MMPCS-VHH-FLAG.gb,No match Found,False
1,None,pET-VHH-FLAG.gb,No match Found,False
2,Var_ABC_LOOP-KGPYGE_SVESLC,Var_ABC_LOOP-KGPYGE_SVESLC.gb,OK,True
3,Var_ABC_LOOP-KNPDGTLT_ANPEYC,Var_ABC_LOOP-KNPDGTLT_ANPEYC.gb,OK,True
4,Var_AB_LOOP-DGPTGE,Var_AB_LOOP-DGPTGE.gb,OK,True
5,Var_AB_LOOP-EVERSGHKVKE,Var_AB_LOOP-EVERSGHKVKE.gb,OK,True
6,Var_AB_LOOP-KGPYGE,Var_AB_LOOP-KGPYGE.gb,OK,True
7,Var_AB_LOOP-KNPDGTLT,Var_AB_LOOP-KNPDGTLT.gb,OK,True
8,Var_AB_LOOP-PATPTSTRGAGGEE,Var_AB_LOOP-PATPTSTRGAGGEE.gb,OK,True
9,Var_AB_LOOP-TDTFPTANWTGEV,Var_AB_LOOP-TDTFPTANWTGEV.gb,OK,True


### Stacked Alignment Results

In [42]:
if verified_records:
    print("\n" + "="*60)
    print("VARIANT ALIGNMENT INSPECTION (Verified Sequences)")
    print("="*60)
    
    # We perform a dumb "Stack" alignment since they should all be identical 
    # except for the loop. We assume they align globally to the first one.
    # Note: If length varies significantly, this visual might shift, but 
    # for loop variants it's usually clear.
    
    # Sort by ID for cleaner reading
    verified_records.sort(key=lambda x: x.id)
    
    # Use Biopython's rudimentary print, or a custom print to focus on the loop
    # Let's align them all to the first one to ensure they stack correctly visually
    if len(verified_records) > 0:
        # We can use the first record as a pivot
        pivot = verified_records[0]
        
        # Determine the "Loop Region" indices based on Upstream Length
        # This helps us 'zoom in' visually if the protein is huge
        start_view = max(0, len(UPSTREAM_SEQ) - 10)
        end_view = start_view + 50 # View 50 AAs around the loop
        
        print(f"Showing alignment region around the loop (Indices {start_view}-{end_view})...")
        print("-" * 80)
        
        for rec in verified_records:
            # Print the ID (padded) and the sequence slice
            seq_str = str(rec.seq)
            # Ensure we don't crash if sequence is short
            slice_end = min(len(seq_str), end_view)
            
            # Highlight loop?
            # We know the loop starts after UPSTREAM_SEQ
            pre = seq_str[:len(UPSTREAM_SEQ)]
            loop_and_post = seq_str[len(UPSTREAM_SEQ):]
            
            # Simple print
            print(f"{rec.id:<25} | {seq_str[start_view:slice_end]}")
            
    print("-" * 80)
    print("(Use this view to verify your loop diversity is correct)")


VARIANT ALIGNMENT INSPECTION (Verified Sequences)
Showing alignment region around the loop (Indices 30-80)...
--------------------------------------------------------------------------------
Var_ABC_LOOP-KGPYGE_SVESLC | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKKGPYGELVYTIK
Var_ABC_LOOP-KNPDGTLT_ANPEYC | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKKNPDGTLTLVYT
Var_AB_LOOP-DGPTGE        | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKDGPTGELVYTIK
Var_AB_LOOP-EVERSGHKVKE   | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEVERSGHKVKEL
Var_AB_LOOP-KGPYGE        | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKKGPYGELVYTIK
Var_AB_LOOP-KNPDGTLT      | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKKNPDGTLTLVYT
Var_AB_LOOP-PATPTSTRGAGGEE | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKPATPTSTRGAGG
Var_AB_LOOP-TDTFPTANWTGEV | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKTDTFPTANWTGE
Var_AB_LOOP-TLPDGSKE      | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKTLPDGSKELVYT
Var_C_LOOP-ANPEYC         | AEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIK
Var_C_LOOP-AS

### Pairwise Alignment Visulaization

In [43]:
# Setup the Aligner
aligner = Align.PairwiseAligner()
aligner.mode = 'global'
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

# 3. Iterate through Verified Results
# We use the DataFrame 'df' from the previous cell
verified_df = df[df['QC_Pass'] == True]

print(f"Comparing {len(verified_df)} verified variants against FULL_CANONICAL\n")

for index, row in verified_df.iterrows():
    variant_name = row['Variant']
    found_seq = row['Found_AA'].rstrip('*')
    canonical_clean = FULL_CANONICAL.rstrip('*')
    
    # Run Alignment
    alignment = aligner.align(canonical_clean, found_seq)[0]
    
    print("="*80)
    print(f"VARIANT: {variant_name}")
    print("Top: Canonical | Bottom: Variant")
    print("-" * 80)
    print(alignment)
    print("\n")

Comparing 15 verified variants against FULL_CANONICAL

VARIANT: Var_ABC_LOOP-KGPYGE_SVESLC
Top: Canonical | Bottom: Variant
--------------------------------------------------------------------------------
target            0 MKVLIVLLAIFAALPLALAQPVISTTVGSAAEGSLDKRCTCSPSHPQDAFCNSDIVIRAK
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 MKVLIVLLAIFAALPLALAQPVISTTVGSAAEGSLDKRCTCSPSHPQDAFCNSDIVIRAK

target           60 VVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEASESLCGLKLEVNKYQYLLT
                 60 ||||||||.||.|.||||||||||||||||||||||||||..||||-|||||||||||||
query            60 VVGKKLVKKGPYGELVYTIKQMKMYRGFTKMPHVQYIHTESVESLC-LKLEVNKYQYLLT

target          120 GRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCNCKIKSCYYLPCFVTSKNECLW
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           119 GRVYDGKMYTGLCNFVERWDQLTLSQRKGLNYRYHLGCNCKIKSCYYLPCFVTSKNECLW

target          180 TDMLSNFGYPGYQSKHYACIRQKGGYCSWYRGWAPPDKSIINA

### Grouped Multi-alignment by Length

In [44]:
print_smart_stacked_alignment(FULL_CANONICAL, df)

Aligning 15 passing variants...

ALIGNMENT GROUP 1/9
Variants in this group: 5

Region 1-150:
FULL_CANONICAL            MKVLIVLLAIFAALPLALAQPVISTTVGSAAEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEASESLCGLKLEVNKYQYLLTGRVYDGKMYTGLCNFVERWDQLTLSQRKGL
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Var_ABC_LOOP-KGPYGE_SV..  ....................................................................K..Y.E..........................SV....-...........................................
Var_AB_LOOP-DGPTGE        ....................................................................D..T.E............................................................................
Var_AB_LOOP-KGPYGE        ....................................................................K..Y.E............................................................................
Var_C_LOOP-ANPEYC    

### Composite Alignment

In [45]:
print_unified_alignment(FULL_CANONICAL, df)

Constructing Unified Alignment for 15 variants...

UNIFIED ALIGNMENT (All Variants vs Canonical)

Region 1-150:
FULL_CANONICAL            MKVLIVLLAIFAALPLALAQPVISTTVGSAAEGSLDKRCTCSPSHPQDAFCNSDIVIRAKVVGKKLVK--------E-----GPFG--T-------LVYTIKQMKMYRGFTKMPHVQYIHTE---------AS-------ESLCGLKLEV
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Var_ABC_LOOP-KGPYGE_SV..  ....................................................................--------K-----..Y.--E-------..........................---------SV-------....-.....
Var_ABC_LOOP-KNPDGTLT_..  ....................................................................--------K-----N.D.TL.-------..........................---------.N-------PEY.-.....
Var_AB_LOOP-DGPTGE        ....................................................................--------D-----..T.--E-------..........................---------..-------..........
Var